# 4.6 Obstacle Labeling Tool

**Usage — restart kernel and run all cells top-to-bottom:**

1. A single widget panel appears (no matplotlib canvas — everything is rendered as PNGs).
2. Use the **Tile** dropdown to switch between tiles.
3. Use the **Cluster** dropdown or **◀ Prev / Next ▶** buttons to navigate clusters.
4. The **map** (left) shows the 2D label overview with the selected cluster marked by a star.
5. The **top + side view** (right) shows the cluster in its scan RGB colours with height info.
6. Use **✔ Accept** to confirm the auto-label, **✎ Update** to save a different label from the dropdown, **⚑ Flag** to mark as uncertain.
7. All decisions are saved to `inventory.csv` immediately. The status icon in the dropdown (`·` / `✔` / `⚑`) updates after each action.

**Map dot borders:** thin = not yet reviewed · white = confirmed · red = flagged

In [ ]:
import sys, io
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import numpy as np
import pandas as pd
import laspy
import ipywidgets as widgets
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

from config import CLUSTERS_DIR, LABELED_DIR

In [ ]:
INV_PATH = CLUSTERS_DIR / 'inventory.csv'
inv = pd.read_csv(INV_PATH)

# Ensure labeling columns exist
for col, default in [('final_label', None), ('label_source_final', None), ('needs_review', False)]:
    if col not in inv.columns:
        inv[col] = default

# Prevent dtype inference from making string columns float64 when all-NaN
inv['label_source_final'] = inv['label_source_final'].astype(object)
inv['final_label'] = inv['final_label'].astype(object)

TILECODES = sorted(inv['tilecode'].unique().tolist())
print(f'Loaded {len(inv)} clusters across {len(TILECODES)} tile(s)')
print(inv.groupby('label')['cluster_idx'].count().rename('count').to_string())

In [ ]:
LABEL_NAMES = {
    0: 'Unknown', 1: 'Road', 9: 'Ground', 10: 'Building',
    30: 'Tree', 40: 'Car', 44: 'Bicycle', 50: 'Person',
    60: 'Street Light', 61: 'Traffic Light', 62: 'Traffic Sign',
    65: 'Bollard', 67: 'Stop Pole', 80: 'City Bench',
    81: 'Rubbish Bin', 83: 'Large Container', 85: 'Parking Meter',
    88: 'Bicycle Rack', 99: 'Noise / False Positive',
}

# Choices available in the label dropdown
LABEL_OPTIONS = [(f"{v}  ({k})", k) for k, v in sorted(LABEL_NAMES.items())]

DOT_COLORS  = {0:'#aaaaaa', 30:'#44ee44', 40:'#ff8800', 60:'#44aaff', 83:'#dd44dd'}
DOT_DEFAULT = '#ffffff'

LABEL_PRIORITY = {10:8, 1:7, 30:6, 40:5, 60:4, 83:4, 79:3, 90:3, 9:2, 0:1}
BG_RGB = {
    -1:(0.07,0.07,0.07),  0:(0.22,0.22,0.22),  1:(0.75,0.20,0.20),
     9:(0.50,0.50,0.50), 10:(0.20,0.35,0.70), 30:(0.20,0.65,0.20),
    40:(1.00,0.50,0.10), 60:(1.00,0.95,0.20), 79:(0.80,0.40,0.00),
    83:(0.70,0.20,0.70), 90:(0.90,0.60,0.10),
}
GRID_RES = 0.25

In [ ]:
# ── tile loading + 2-D background grid ────────────────────────────────────────
_tile_cache = {}
_bg_cache   = {}

def _load_tile_raw(tilecode):
    if tilecode in _tile_cache:
        return _tile_cache[tilecode]
    laz_path = LABELED_DIR / f'bgt_labeled_{tilecode}.laz'
    print(f'  Loading {laz_path.name}…', end=' ', flush=True)
    pc  = laspy.read(laz_path)
    xy  = np.column_stack([np.asarray(pc.x, np.float32),
                           np.asarray(pc.y, np.float32)])
    has = 'label' in pc.point_format.extra_dimension_names
    lbl = np.asarray(pc.label, np.int32) if has else np.zeros(len(xy), np.int32)
    print(f'{len(xy):,} pts')
    _tile_cache[tilecode] = (xy, lbl)
    return xy, lbl

def _make_bg(xy, labels):
    x, y   = xy[:,0], xy[:,1]
    x0, y0 = float(x.min()), float(y.min())
    xi = np.floor((x - x0) / GRID_RES).astype(np.int32)
    yi = np.floor((y - y0) / GRID_RES).astype(np.int32)
    nx, ny = int(xi.max())+1, int(yi.max())+1
    order  = np.argsort(np.vectorize(lambda l: LABEL_PRIORITY.get(int(l), 0))(labels))
    grid   = np.full((ny, nx), -1, np.int32)
    grid[yi[order], xi[order]] = labels[order]
    rgb    = np.full((ny, nx, 3), BG_RGB[-1], np.float32)
    for lv, c in BG_RGB.items():
        m = grid == lv
        if m.any(): rgb[m] = c
    return rgb, [x0, x0 + nx*GRID_RES, y0, y0 + ny*GRID_RES]

def _get_bg(tilecode):
    if tilecode not in _bg_cache:
        xy, lbl = _load_tile_raw(tilecode)
        _bg_cache[tilecode] = _make_bg(xy, lbl)
    return _bg_cache[tilecode]

In [ ]:
# ── detail renderer (RGB colours from scan, height range in title) ─────────────
def _sa(ax):
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
    ax.tick_params(labelsize=6, colors='grey')

def _fig_bytes(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, facecolor='#1a1a1a', bbox_inches='tight')
    buf.seek(0)
    return buf.read()

def render_detail(row):
    """Top-view (XY) + side-view (XZ) coloured by scan RGB, height range in title."""
    try:
        npz = np.load(row['npz_path'])
    except Exception as e:
        fig = Figure(figsize=(9, 4), facecolor='#1a1a1a')
        FigureCanvasAgg(fig)
        ax = fig.add_subplot(111); _sa(ax)
        ax.text(0.5, 0.5, f'Cannot load NPZ:\n{e}', color='#cc4444',
                ha='center', va='center', transform=ax.transAxes)
        return _fig_bytes(fig)

    xyz    = npz['xyz_centered']
    colors = np.clip(npz['rgb_norm'], 0, 1)
    h      = npz['height_ag']
    pt_sz  = max(1, min(10, 3000 // max(len(xyz), 1)))

    lbl   = int(row['label'])
    flbl  = row.get('final_label', None)
    name  = LABEL_NAMES.get(lbl, f'Label {lbl}')
    fname = LABEL_NAMES.get(int(flbl), f'Label {flbl}') if pd.notna(flbl) else 'not reviewed'
    title = (f"#{int(row['cluster_idx'])}  auto: {name} ({lbl})  ·  confirmed: {fname}"
             f"  ·  {int(row['n_raw_pts']):,} pts  ·  {float(row['area_m2']):.2f} m²"
             f"  ·  {float(h.min()):.1f}–{float(h.max()):.1f} m above ground")

    fig = Figure(figsize=(10, 4.5), facecolor='#1a1a1a')
    FigureCanvasAgg(fig)
    fig.suptitle(title, color='white', fontsize=8)
    ax_t = fig.add_subplot(1, 2, 1)
    ax_s = fig.add_subplot(1, 2, 2)
    _sa(ax_t); _sa(ax_s)

    ax_t.scatter(xyz[:,0], xyz[:,1], c=colors, s=pt_sz, linewidths=0)
    ax_t.set_aspect('equal')
    ax_t.set_title('top view (XY)', color='#aaaaaa', fontsize=8)
    ax_t.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_t.set_ylabel('ΔY (m)', color='grey', fontsize=7)

    ax_s.scatter(xyz[:,0], xyz[:,2], c=colors, s=pt_sz, linewidths=0)
    ax_s.set_aspect('equal')
    ax_s.set_title('side view (XZ)', color='#aaaaaa', fontsize=8)
    ax_s.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_s.set_ylabel('height (m)', color='grey', fontsize=7)

    fig.tight_layout()
    return _fig_bytes(fig)

In [ ]:
# ── 2D overview map renderer (Agg PNG — no %matplotlib widget) ────────────────
def render_map(tilecode, selected_inv_idx=None):
    """
    Returns PNG bytes: 2D label-grid overview with all cluster dots.
    The selected cluster (if given) is highlighted with a white star.
    """
    rgb, extent = _get_bg(tilecode)
    tile_inv    = inv[inv['tilecode'] == tilecode]

    fig = Figure(figsize=(5.5, 5.5), facecolor='#111')
    FigureCanvasAgg(fig)
    ax  = fig.add_subplot(111)
    ax.set_facecolor('#111')
    for sp in ax.spines.values():
        sp.set_edgecolor('#444')
    ax.tick_params(colors='#777', labelsize=7)

    ax.imshow(rgb, origin='lower', extent=extent,
              interpolation='nearest', aspect='equal')
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_title(tilecode, color='white', fontsize=9)
    ax.set_xlabel('X (m RD)', color='#777', fontsize=7)
    ax.set_ylabel('Y (m RD)', color='#777', fontsize=7)

    for lbl_code, grp in tile_inv.groupby('label'):
        base_c   = DOT_COLORS.get(lbl_code, DOT_DEFAULT)
        reviewed = grp['final_label'].notna().values
        flagged  = grp['needs_review'].fillna(False).astype(bool).values
        ec = ['#ff3333' if f else ('#ffffff' if r else '#444444')
              for r, f in zip(reviewed, flagged)]
        ew = [1.8 if f else (1.5 if r else 0.5)
              for r, f in zip(reviewed, flagged)]
        ax.scatter(
            grp['centroid_x'].values, grp['centroid_y'].values,
            c=base_c, s=80, zorder=5,
            label=LABEL_NAMES.get(lbl_code, str(lbl_code)),
            edgecolors=ec, linewidths=ew,
        )

    if selected_inv_idx is not None and selected_inv_idx in inv.index:
        row = inv.loc[selected_inv_idx]
        ax.plot(row['centroid_x'], row['centroid_y'], 'w*',
                markersize=22, zorder=10,
                markeredgecolor='#ff3333', markeredgewidth=1.5)

    ax.legend(facecolor='#1e1e1e', labelcolor='white',
              edgecolor='#444', fontsize=7, loc='upper right')
    fig.tight_layout()
    return _fig_bytes(fig)

In [ ]:
# ── labeling helpers ───────────────────────────────────────────────────────────
def _save_inv():
    inv.to_csv(INV_PATH, index=False)

def _progress_str(tilecode):
    tile = inv[inv['tilecode'] == tilecode]
    reviewed = tile['final_label'].notna().sum()
    flagged  = tile['needs_review'].fillna(False).sum()
    return (f'<span style="color:#aaa;font-size:12px">'f'Tile: '
            f'<b>{reviewed}/{len(tile)}</b> reviewed'
            f'  ·  <span style="color:#ff6666">{flagged} flagged</span></span>')

def _dot_style(row):
    """Return (facecolor, edgecolor, edgewidth, alpha) for a cluster dot."""
    color = DOT_COLORS.get(int(row['label']), DOT_DEFAULT)
    if row.get('needs_review', False):
        return color, '#ff3333', 1.8, 1.0   # flagged: red edge
    if pd.notna(row.get('final_label', None)):
        return color, '#ffffff', 1.5, 1.0   # confirmed: white edge
    return color, '#111111', 0.5, 0.65      # unreviewed: dim, thin edge

In [ ]:
# ── All-Agg labeling UI — no %matplotlib widget ────────────────────────────────

# ── cluster dropdown options ───────────────────────────────────────────────────
def _cluster_options(tilecode):
    tile = inv[inv['tilecode'] == tilecode]
    opts = []
    for _, r in tile.iterrows():
        lname  = LABEL_NAMES.get(int(r['label']), str(r['label']))
        flbl   = r.get('final_label', None)
        nr     = bool(r.get('needs_review', False))
        status = '✔' if pd.notna(flbl) else ('⚑' if nr else '·')
        label  = (f"#{int(r['cluster_idx']):>3}  {lname:<18}"
                  f"  {int(r['n_raw_pts']):>7,} pts  {status}")
        opts.append((label, r.name))   # value = original inv.index
    return opts


# ── widgets ────────────────────────────────────────────────────────────────────
tile_dd = widgets.Dropdown(
    options=TILECODES, value=TILECODES[0],
    description='Tile:',
    layout=widgets.Layout(width='280px'),
    style={'description_width': '40px'},
)
progress_html = widgets.HTML(value='')

cluster_dd = widgets.Dropdown(
    options=_cluster_options(TILECODES[0]),
    description='Cluster:',
    layout=widgets.Layout(width='440px'),
    style={'description_width': '60px'},
)
btn_prev = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
btn_next = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))

info_html = widgets.HTML(value='')

label_dd = widgets.Dropdown(
    options=LABEL_OPTIONS, value=0,
    description='Label:',
    layout=widgets.Layout(width='300px'),
    style={'description_width': '50px'},
)
btn_accept = widgets.Button(description='✔ Accept', button_style='success',
                            layout=widgets.Layout(width='105px'),
                            tooltip='Confirm the auto-label as-is')
btn_update = widgets.Button(description='✎ Update', button_style='primary',
                            layout=widgets.Layout(width='105px'),
                            tooltip='Save the dropdown label as final')
btn_flag   = widgets.Button(description='⚑ Flag',   button_style='warning',
                            layout=widgets.Layout(width='105px'),
                            tooltip='Mark as uncertain / needs review')

map_img    = widgets.Image(value=b'', format='png',
                           layout=widgets.Layout(width='560px'))
detail_img = widgets.Image(value=b'', format='png',
                           layout=widgets.Layout(width='700px'))


# ── refresh helpers ────────────────────────────────────────────────────────────
def _refresh_detail(idx):
    row   = inv.loc[idx]
    lbl   = int(row['label'])
    flbl  = row.get('final_label', None)
    fname = LABEL_NAMES.get(int(flbl), str(flbl)) if pd.notna(flbl) else '—'
    nr    = bool(row.get('needs_review', False))
    info_html.value = (
        f'<span style="color:#ccc;font-size:13px">'
        f'<b>#{int(row["cluster_idx"])}  {LABEL_NAMES.get(lbl, str(lbl))}</b>'
        f' · {row.get("label_source", "")}'
        f' · {int(row["n_raw_pts"]):,} pts · {float(row["area_m2"]):.2f} m²'
        f'<br>confirmed: <b>{fname}</b>'
        + ('  <span style="color:#ff6666">⚑ flagged</span>' if nr else '')
        + '</span>'
    )
    target = int(flbl) if pd.notna(flbl) else lbl
    label_dd.value = target if target in dict(LABEL_OPTIONS).values() else 0
    detail_img.value = render_detail(row)

def _refresh_map(idx):
    map_img.value = render_map(tile_dd.value, selected_inv_idx=idx)

def _refresh(idx):
    _refresh_map(idx)
    _refresh_detail(idx)


# ── navigation ─────────────────────────────────────────────────────────────────
def _on_cluster(change):
    if change['name'] == 'value' and change['new'] is not None:
        _refresh(change['new'])

def _on_prev(_):
    vals = [o[1] for o in cluster_dd.options]
    i = vals.index(cluster_dd.value)
    if i > 0:
        cluster_dd.value = vals[i - 1]

def _on_next(_):
    vals = [o[1] for o in cluster_dd.options]
    i = vals.index(cluster_dd.value)
    if i < len(vals) - 1:
        cluster_dd.value = vals[i + 1]

cluster_dd.observe(_on_cluster)
btn_prev.on_click(_on_prev)
btn_next.on_click(_on_next)


# ── tile switch ────────────────────────────────────────────────────────────────
def _on_tile(change):
    if change['name'] == 'value':
        opts = _cluster_options(change['new'])
        cluster_dd.options = opts
        cluster_dd.value   = opts[0][1]
        progress_html.value = _progress_str(change['new'])

tile_dd.observe(_on_tile)


# ── labeling ───────────────────────────────────────────────────────────────────
def _apply_label(final_lbl, source, flag=False):
    idx = cluster_dd.value
    if idx is None or idx not in inv.index:
        info_html.value = '<span style="color:#c44">No cluster selected</span>'
        return
    inv.at[idx, 'final_label']        = final_lbl
    inv.at[idx, 'label_source_final'] = source
    inv.at[idx, 'needs_review']       = flag
    _save_inv()
    # Rebuild dropdown so status icon updates; stay on the same cluster
    cur = idx
    cluster_dd.options  = _cluster_options(tile_dd.value)
    cluster_dd.value    = cur
    progress_html.value = _progress_str(tile_dd.value)
    _refresh_detail(cur)   # detail only — map re-renders on next tile switch

def _on_accept(_):
    idx = cluster_dd.value
    if idx is not None and idx in inv.index:
        _apply_label(int(inv.loc[idx, 'label']), 'confirmed')

def _on_update(_):
    _apply_label(label_dd.value, 'manual')

def _on_flag(_):
    idx = cluster_dd.value
    if idx is None or idx not in inv.index:
        return
    flbl = inv.at[idx, 'final_label']
    lbl  = int(flbl) if pd.notna(flbl) else int(inv.at[idx, 'label'])
    _apply_label(lbl, 'manual', flag=True)

btn_accept.on_click(_on_accept)
btn_update.on_click(_on_update)
btn_flag.on_click(_on_flag)


# ── layout ─────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([tile_dd, progress_html],
                 layout=widgets.Layout(gap='20px', align_items='center')),
    widgets.HBox([cluster_dd, btn_prev, btn_next],
                 layout=widgets.Layout(gap='8px', align_items='center')),
    widgets.HBox([
        map_img,
        widgets.VBox([
            info_html,
            widgets.HBox([label_dd, btn_accept, btn_update, btn_flag],
                         layout=widgets.Layout(gap='6px')),
            detail_img,
        ], layout=widgets.Layout(padding='0 0 0 16px')),
    ], layout=widgets.Layout(gap='12px', align_items='flex-start')),
]))


# ── initial render ─────────────────────────────────────────────────────────────
progress_html.value = _progress_str(TILECODES[0])
if cluster_dd.options:
    _refresh(cluster_dd.options[0][1])